# Modelos de decisión tipada como guardarraíles: el ataque del canal de opciones

Cuaderno de lectura de la medición `guardarrailes/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Réplica de arXiv 2610.12292 sobre LAYA-TD y tres modelos de la semana del 5 de octubre de 2026 (d1-omni-600M, d1-3B y LemonDecider), y el recuento de la ola de estos modelos en Hugging Face. Carga los ficheros de al lado (o los descarga del repositorio si se ejecuta fuera de él), muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no repite la medición: para eso está `generar.py`.

*Reading notebook for this measurement: loads the files next to it, prints the provenance record and draws one figure. It only reads; `generar.py` repeats the measurement.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/guardarrailes/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")

## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
r = leer("resumen_condiciones.csv")
r["acierto"] = (100 * r.accuracy).round(1)
r["fallo_abierto"] = (100 * r.fail_open).round(1)
r["fallo_cerrado"] = (100 * r.fail_closed).round(1)
tabla = r.pivot(index="condicion", columns="modelo",
                values=["acierto", "fallo_abierto", "fallo_cerrado"])
print(tabla.to_string())
print()
b = leer("presupuesto_barrido.csv")
print(b[["modelo", "honesta", "k1", "k5", "k10", "mejor", "mejor_etiqueta"]]
      .round(3).to_string(index=False))
print()
print(leer("entrada_etiqueta.csv").to_string(index=False))
print()
print(leer("ola_semanal.csv").tail(6).to_string(index=False))

## Una figura

In [ ]:
orden = ["laya-td", "d1-omni-600m", "d1-3b", "lemon"]
conds = {"honesta": "nombres originales", "permisiva_escalate": "«allow» → «escalate»",
         "registro_k6": "6 líneas de registro"}
fig, ax = plt.subplots(figsize=(9, 4.2))
ancho = 0.26
for i, (c, rotulo) in enumerate(conds.items()):
    v = r[r.condicion == c].set_index("modelo").loc[orden, "fallo_abierto"]
    ax.bar([x + (i - 1) * ancho for x in range(len(orden))], v, ancho, label=rotulo)
ax.set_xticks(range(len(orden)), orden)
ax.set_ylabel("fallo abierto (%)")
ax.set_title("Llamadas prohibidas que el guardarraíl deja pasar (1.800 por celda)")
ax.legend(frameon=False)
plt.tight_layout()